<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Bölüm 7 Alıştırma Çözümleri

## Alıştırma 7.1: İstem biçimlerini değiştirmek

Elimizde şöyle bir veri kaydı olduğunu varsayalım:

```json
{
  "instruction": "Identify the correct spelling of the following word.",
  "input": "Ocassion",
  "output": "The correct spelling is 'Occasion.'"
}
```

Ana bölümde bunu Alpaca tarzı istem şablonuna göre biçimlendirmiştik:

```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Occassion

### Response:
The correct spelling is 'Occasion.'
```

Bu alıştırmada ise onun yerine Phi-3 istem şablonunu kullanıyoruz; bu şablon veri kaydını şöyle biçimlendirir:

```
<user>
Identify the correct spelling of the following word: 'Occasion'

<assistant>
The correct spelling is 'Occasion'.
```

Bu istem şablonunun belirgin biçimde daha kısa olduğunu unutmayın; girdi istemleri kısaldığı için LLM'e ince ayar yapmanın ve metin üretmenin çalışma süresi ile donanım gereksinimleri de azalır.
Bu değişikliği yapmak için `format_input` fonksiyonunu şöyle güncelliyoruz:

In [1]:
def format_input(entry):
    instruction_text = (
        f"<|user|>\n{entry['instruction']}"
    )

    input_text = f"\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

Biri `'input'` alanında içerik olan, diğeri olmayan iki örnek girdiye uygulayarak beklendiği gibi çalıştığından emin olalım:

In [2]:
sample_data = [
    {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}, 
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}
]

print(format_input(sample_data[0]))
print()
print(format_input(sample_data[1]))

<|user|>
Identify the correct spelling of the following word.
Ocassion

<|user|>
What is an antonym of 'complicated'?


Ardından, yanıt için <|assistant|> istem şablonunu kullanmak üzere `InstructionDataset` sınıfını da güncelliyoruz:

```python
import tiktoken
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Pre-tokenize texts
        self.encoded_texts = []
        for entry in data:

            ###################################################################
            # NEW: Use `format_input_phi` and adjust the response text template
            instruction_plus_input = format_input(entry)
            response_text = f"\n<|assistant|>:\n{entry['output']}"
            ###################################################################
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


tokenizer = tiktoken.get_encoding("gpt2")
```

Son olarak, test kümesi yanıtlarını toplarken üretilen yanıtı ayıklama biçimimizi de güncellememiz gerekiyor:

```python
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

    input_text = format_input(entry)
    tokenizer=tokenizer

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)

    # New: Adjust ###Response -> <|assistant|>
    response_text = generated_text[len(input_text):].replace("<|assistant|>:", "").strip()

    test_data[i]["model_response"] = response_text
```

Kolaylık olsun diye alıştırma çözümü [exercise_experiments.py](exercise_experiments.py) betiğinde uygulanmıştır; şöyle çalıştırabilirsiniz:

```bash
python exercise_experiments.py --exercise_solution phi3_prompt
```

Çıktı:

```
matplotlib version: 3.7.1
tiktoken version: 0.7.0
torch version: 2.3.0+cu121
tqdm version: 4.66.4
tensorflow version: 2.15.0
--------------------------------------------------
Training set length: 935
Validation set length: 55
Test set length: 110
--------------------------------------------------
Device: cuda
--------------------------------------------------
...
Loaded model: gpt2-medium (355M)
--------------------------------------------------
Initial losses
   Training loss: 3.71630220413208
   Validation loss: 3.6440994262695314
Ep 1 (Step 000000): Train loss 2.633, Val loss 2.622
...
Ep 2 (Step 000230): Train loss 0.424, Val loss 0.928
<|user|> Convert the active sentence to passive: 'The chef cooks the meal every day.' <|assistant|>: The meal is prepared every day by the chef....
Training completed in 1.50 minutes.
Plot saved as loss-plot-phi3-prompt.pdf
--------------------------------------------------
Generating responses
100% 110/110 [00:11<00:00,  9.27it/s]
Responses saved as instruction-data-with-response-phi3-prompt.json
Model saved as gpt2-medium355M-sft-phi3-prompt.pth
```

Karşılaştırma için, özgün 7. bölüm ince ayar kodunu `python exercise_experiments.py --exercise_solution baseline` ile çalıştırabilirsiniz. 

Bir Nvidia L4 GPU'da, Phi-3 istem şablonunu kullanan yukarıdaki kodun 1,5 dakikada çalıştığını unutmayın. Buna karşılık Alpaca tarzı şablon 1,80 dakika sürüyor. Yani Phi-3 şablonu, daha kısa model girdileri ürettiği için yaklaşık %17 daha hızlı. 

Doğru biçimlendirildiklerinden emin olmak için yanıtlardan bazılarına göz atalım:

```json
    {
        "instruction": "Rewrite the sentence using a simile.",
        "input": "The car is very fast.",
        "output": "The car is as fast as lightning.",
        "model_response": "The car is as fast as a cheetah."
    },
    {
        "instruction": "What type of cloud is typically associated with thunderstorms?",
        "input": "",
        "output": "The type of cloud typically associated with thunderstorms is cumulonimbus.",
        "model_response": "The type of cloud associated with thunderstorms is a cumulus cloud."
    },
    {
        "instruction": "Name the author of 'Pride and Prejudice'.",
        "input": "",
        "output": "Jane Austen.",
        "model_response": "The author of 'Pride and Prejudice' is Jane Austen."
    },
```

Başarımı Ollama Llama 3 yöntemiyle değerlendirebiliriz; kolaylık olsun diye bu da `python exercise_experiments.py` betiğinde uygulanmıştır ve şöyle çalıştırılabilir:

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-phi3-prompt.json
```

Çıktı:

```
Ollama running: True
Scoring entries: 100%|████████████████████████| 110/110 [01:08<00:00,  1.60it/s]
Number of scores: 110 of 110
Average score: 48.87
```

Puan 50'ye yakın; bu da Alpaca tarzı istemlerle daha önce elde ettiğimiz puanla aynı aralıkta.

Phi istem biçiminin daha iyi olmasını gerektiren doğuştan bir üstünlük ya da gerekçe yok; ancak aşağıdaki *İpucu* bölümünde belirtilen çekince dışında daha derli toplu ve verimli olabilir.

#### İpucu: Özel token'ları göz önünde bulundurmak

- Phi-3 istem şablonunun `<|user|>` ve `<|assistant|>` gibi özel token'lar içerdiğini unutmayın; bunlar GPT-2 tokenizer'ı için ideal olmayabilir
- GPT-2 tokenizer'ı `<|endoftext|>` ifadesini özel bir token olarak tanısa da (50256 token kimliğine kodlar), yukarıda söz edilenler gibi diğer özel token'ları işlemede verimsizdir
- Örneğin `<|user|>` ifadesi 5 ayrı token kimliğine (27, 91, 7220, 91, 29) kodlanır; bu çok verimsizdir
- `<|user|>` ifadesini `allowed_special` argümanı aracılığıyla `tiktoken` içinde yeni bir özel token olarak ekleyebiliriz; ancak GPT-2 sözcük dağarcığının ek bir değişiklik olmadan bunu işleyemeyeceğini unutmayın
- Bir tokenizer'ın ve LLM'in özel token'ları işleyecek şekilde nasıl genişletilebileceğini merak ediyorsanız [extend-tiktoken.ipynb](../../ch05/09_extending-tokenizers/extend-tiktoken.ipynb) bonus materyaline bakın (burada gerekli olmadığını, yalnızca meraklı okurlar için ilginç bir bonus değerlendirme olduğunu unutmayın)
- Ayrıca, bir istem şablonundaki bu özel token'ları sözcük dağarcığı üzerinden destekleyen modellerin genel olarak daha verimli ve daha iyi başarım göstereceğini varsayabiliriz

&nbsp;
## Alıştırma 7.2: Talimat ve girdi maskeleme

Talimatları aşağıdaki şekilde gösterildiği gibi maskelemek için `InstructionDataset` sınıfında ve `custom_collate_fn` fonksiyonunda küçük değişiklikler yapmamız gerekiyor.

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/mask-instructions.webp" width=600px>

In [4]:
# Bu `format_input` fonksiyonu özgün 7. bölüm kodundan kopyalanmıştır

def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

`InstructionDataset` sınıfını, talimatların uzunluklarını toplayacak şekilde değiştirebiliriz; collate fonksiyonunu kodlarken bu uzunlukları hedefler içindeki talimat içeriğinin konumunu bulmak için kullanacağız:

In [5]:
import torch
from torch.utils.data import Dataset


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        ##########################################################################################
        # New: Separate list for instruction lengths
        self.instruction_lengths = []
        ##########################################################################################
        
        self.encoded_texts = []
        
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

            ##########################################################################################
            # New: collect instruction lengths
            instruction_length = len(tokenizer.encode(instruction_plus_input))
            self.instruction_lengths.append(instruction_length)
            ##########################################################################################
            
    def __getitem__(self, index):
        # New: return both instruction lengths and texts separately
        return self.instruction_lengths[index], self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [6]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

Ardından `custom_collate_fn` fonksiyonunu güncelliyoruz; `InstructionDataset` veri kümesindeki değişiklikler nedeniyle artık her `batch`, yalnızca `item` yerine `(instruction_length, item)` içeren bir demettir. Ayrıca artık hedef kimlik listesindeki ilgili talimat token'larını maskeliyoruz.

In [7]:
def custom_collate_fn(
    batch,
    pad_token_id=50256,
    ignore_index=-100,
    allowed_max_length=None,
    device="cpu"
):
    # Yığındaki en uzun diziyi bul
    batch_max_length = max(len(item)+1 for instruction_length, item in batch)   # New: batch is now a tuple

    # Girdileri ve hedefleri doldur ve hazırla
    inputs_lst, targets_lst = [], []

    for instruction_length, item in batch:  # New: batch is now a tuple
        new_item = item.copy()
        # Bir <|endoftext|> token'ı ekle
        new_item += [pad_token_id]
        # Dizileri max_length uzunluğuna doldur
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])  # Truncate the last token for inputs
        targets = torch.tensor(padded[1:])  # Shift +1 to the right for targets

        # Hedeflerde ilk dolgu token'ı dışındaki tümünü ignore_index ile değiştir
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        ##########################################################################################
        # New: Mask all input and instruction tokens in the targets
        targets[:instruction_length-1] = -100
        ##########################################################################################
        
        # İsteğe bağlı olarak maksimum dizi uzunluğuna kırp
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]
        
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Girdi ve hedef listelerini tensörlere dönüştür ve hedef cihaza aktar
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

Aşağıda bunu bazı örnek veriler üzerinde deneyelim:

In [8]:
sample_data = [
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."},
    {'instruction': 'Sort the following list in alphabetical order.', 'input': 'Zebra, Elephant, Crocodile', 'output': 'Crocodile, Elephant, Zebra'},
    {'instruction': 'Arrange the given numbers in descending order.', 'input': '5, 12, 8, 3, 15', 'output': '15, 12, 8, 5, 3.'}
]

In [9]:
from torch.utils.data import DataLoader

train_dataset = InstructionDataset(sample_data, tokenizer)
train_loader = DataLoader(
    train_dataset,
    batch_size=len(sample_data),
    collate_fn=custom_collate_fn,
    num_workers=0
)

In [10]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([3, 64]) torch.Size([3, 64])


In [11]:
print("Inputs:\n", inputs[1])
print("\n\nTargets:\n", targets[1])

Inputs:
 tensor([21106,   318,   281, 12064,   326,  8477,   257,  4876,    13, 19430,
          257,  2882,   326, 20431, 32543,   262,  2581,    13,   198,   198,
        21017, 46486,    25,   198, 42758,   262,  1708,  1351,   287, 24830,
          605,  1502,    13,   198,   198, 21017, 23412,    25,   198,    57,
        37052,    11, 42651,    11,  9325, 19815,   576,   198,   198, 21017,
        18261,    25,   198,    34, 12204,   375,   576,    11, 42651,    11,
         1168, 37052, 50256, 50256])


Targets:
 tensor([ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,   198,   198, 21017, 18261,
           25,   198,    34, 12204,   375,   576,    11, 42651,    11,  1168,
      

`targets` tensöründen görebileceğimiz gibi, artık hem talimat hem de dolgu token'ları -100 yer tutucu token'larıyla maskelenmiş durumda. 
Doğru göründüklerinden emin olmak için girdilerin kodunu çözelim:

In [12]:
print(tokenizer.decode(list(inputs[1])))

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Sort the following list in alphabetical order.

### Input:
Zebra, Elephant, Crocodile

### Response:
Crocodile, Elephant, Zebra<|endoftext|><|endoftext|>


Ardından, maskelenmemiş hedef token kimliklerinin kodunu çözelim:

In [13]:
non_masked_targets = targets[1][targets[1] != -100]

print(tokenizer.decode(list(non_masked_targets)))



### Response:
Crocodile, Elephant, Zebra<|endoftext|>


Yukarıda gösterildiği gibi, maskelenmemiş hedef token'lar amaçlandığı üzere `"Instruction"` ve `"Input"` alanlarını dışarıda bırakıyor. Şimdi, bu maskeleme stratejisiyle ince ayar yapıldığında LLM'in ne kadar iyi başarım gösterdiğini görmek için değiştirilmiş kodu çalıştırabiliriz.

Kolaylık olsun diye, bir karşılaştırma çalıştırmak için `exercise_experiments.py` kodunu şöyle kullanabilirsiniz:

```bash
python exercise_experiments.py --exercise_solution mask_instructions
```

Çıktı:

```
matplotlib version: 3.7.1
tiktoken version: 0.7.0
torch version: 2.3.0+cu121
tqdm version: 4.66.4
tensorflow version: 2.15.0
--------------------------------------------------
Training set length: 935
Validation set length: 55
Test set length: 110
--------------------------------------------------
Device: cuda
--------------------------------------------------
...
Loaded model: gpt2-medium (355M)
--------------------------------------------------
Initial losses
   Training loss: 2.280539035797119
   Validation loss: 2.262560224533081
Ep 1 (Step 000000): Train loss 1.636, Val loss 1.620
...
Ep 2 (Step 000230): Train loss 0.143, Val loss 0.727
...
Training completed in 1.77 minutes.
Plot saved as loss-plot-mask-instructions.pdf
--------------------------------------------------
Generating responses
100% 110/110 [02:10<00:00,  1.19s/it]
Responses saved as instruction-data-with-response-mask-instructions.json
Model saved as gpt2-medium355M-sft-mask-instructions.pth
```

Ardından, ortaya çıkan LLM'in başarımını değerlendirelim:

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-mask-instructions.json
```

```
Ollama running: True
Scoring entries: 100%|██████████████████████████████████████████████████████████████████████████████████████| 110/110 [01:23<00:00,  1.31it/s]
Number of scores: 110 of 110
Average score: 47.73
```

Puanlardan görebileceğimiz gibi, talimat maskeleme biraz daha kötü başarım gösteriyor; bu da "Instruction Tuning With Loss Over Instructions" makalesindeki (https://arxiv.org/abs/2405.14394) gözlemle tutarlı

&nbsp;
## Alıştırma 7.3: Özgün Alpaca veri kümesi üzerinde ince ayar

Modele özgün Stanford Alpaca veri kümesi ([https://github.com/tatsu-lab/stanford_alpaca](https://github.com/tatsu-lab/stanford_alpaca)) üzerinde ince ayar yapmak için dosya URL'sini şu satırdan

```python
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
```

şu satıra değiştirmeniz yeterlidir

```python
url = "https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json"
```

Veri kümesinin 52 bin kayıt içerdiğini (7. bölümdekinin 50 katı) ve kayıtların 7. bölümde çalıştıklarımızdan daha uzun olduğunu unutmayın.
Bu nedenle eğitimin bir GPU üzerinde çalıştırılması kesinlikle önerilir.

Bellek yetersizliği hataları alırsanız yığın boyutunu 8'den 4, 2 veya 1'e düşürmeyi değerlendirin. Yığın boyutunu düşürmenin yanı sıra `allowed_max_length` değerini 1024'ten 512 ya da 256'ya indirmeyi de düşünebilirsiniz.

Kolaylık olsun diye, modele 52 bin kayıtlık Alpaca veri kümesi üzerinde 4 yığın boyutu ve 512 `allowed_max_length` değeriyle ince ayar yapmak için `exercise_experiments.py` kodunu şöyle kullanabilirsiniz:

```bash
python exercise_experiments.py --exercise_solution alpaca_52k
```

```
matplotlib version: 3.7.1
tiktoken version: 0.7.0
torch version: 2.3.0+cu121
tqdm version: 4.66.4
tensorflow version: 2.15.0
--------------------------------------------------
Training set length: 44201
Validation set length: 2601
Test set length: 5200
--------------------------------------------------
Device: cuda
--------------------------------------------------
...
Loaded model: gpt2-medium (355M)
--------------------------------------------------
Initial losses
   Training loss: 3.3681655883789063
   Validation loss: 3.4122894287109373
Ep 1 (Step 000000): Train loss 2.477, Val loss 2.750
...
Ep 2 (Step 022095): Train loss 0.761, Val loss 1.557
...
Training completed in 196.38 minutes.
Plot saved as loss-plot-alpaca52k.pdf
--------------------------------------------------
Generating responses
100% 5200/5200 [2:56:33<00:00,  2.04s/it]
Responses saved as instruction-data-with-response-alpaca52k.json
Model saved as gpt2-medium355M-sft-alpaca52k.pth
```

Aşağıda, üretilen model yanıtlarıyla birlikte Alpaca veri kümesinden birkaç örnek yer alıyor:

```json
[
    {
        "instruction": "Edit the following sentence to increase readability: \"He made a huge effort and was so successful.\"",
        "input": "",
        "output": "He exerted a tremendous effort, and thus enjoyed great success.",
        "model_response": "He put in an immense effort and was rewarded with success."
    },
    {
        "instruction": "Rewrite the following sentence to make it more concise: \"I was displeased with the result of the experiment that I conducted.\"",
        "input": "",
        "output": "I was unhappy with my experiment's outcome.",
        "model_response": "I was displeased with the results of the experiment."
    },
    {
        "instruction": "How can we build a more efficient GPT model?",
        "input": "",
        "output": "We can build a more efficient GPT model by optimizing the architecture of the model, using smaller model sizes and training with fewer parameters. We can also leverage techniques such as knowledge distillation, transfer learning, dynamic sparsity and hybrid computing to further improve the efficiency of the model.",
        "model_response": "Building a more efficient GPT model requires careful planning and optimization. First, it is important to identify the target language and the context in which the model is used. Then, it is important to select the appropriate model architecture, such as backpropagation, hyperparameters, and hyperparameters. Finally, it is important to select the appropriate model weights and optimizers, such as backpropagation, hyperparameters, and hyperparameters."
    },
]

Son olarak, ince ayarlı LLM'i [ollama_evaluate.py](ollama_evaluate.py) yardımcı fonksiyonuyla değerlendirebiliriz:

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-alpaca52k.json
```

```
Scoring entries: 100%|████████████████████| 5200/5200 [1:07:52<00:00, 1.28it/s]
Number of scores: 5188 of 5200
Average score: 48.16
```

Puan, bu bölümde kullandığımız veri kümesinde elde ettiğimiz puandan biraz daha düşük. Ancak Alpaca test kümesinin, ana bölümde kullandığımız veri kümesine göre daha çeşitli ve kısmen daha zorlayıcı talimatlar içerdiğini unutmayın.

## Alıştırma 7.4: LoRA ile parametre açısından verimli ince ayar

Modele LoRA kullanarak talimat ince ayarı yapmak için Ek E'deki ilgili sınıf ve fonksiyonları kullanın:

```python
from appendix_E import LoRALayer, LinearWithLoRA, replace_linear_with_lora
```

Ardından, 7.5 kısmındaki model yükleme kodunun altına şu kod satırlarını ekleyin:


```python
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params:,}")

for param in model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters after: {total_params:,}")
replace_linear_with_lora(model, rank=16, alpha=16)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")
model.to(device)
```

Kolaylık olsun diye, modele 16 rütbeli (rank) ve 16 alfa değerli LoRA kullanarak ince ayar yapmak için `exercise_experiments.py` kodunu şöyle kullanabilirsiniz:

```bash
python exercise_experiments.py --exercise_solution lora
```

Çıktı:

```
matplotlib version: 3.7.1
tiktoken version: 0.7.0
torch version: 2.3.0+cu121
tqdm version: 4.66.4
tensorflow version: 2.15.0
--------------------------------------------------
Training set length: 935
Validation set length: 55
Test set length: 110
--------------------------------------------------
Device: cuda
--------------------------------------------------
File already exists and is up-to-date: gpt2/355M/checkpoint
File already exists and is up-to-date: gpt2/355M/encoder.json
File already exists and is up-to-date: gpt2/355M/hparams.json
File already exists and is up-to-date: gpt2/355M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/355M/model.ckpt.index
File already exists and is up-to-date: gpt2/355M/model.ckpt.meta
File already exists and is up-to-date: gpt2/355M/vocab.bpe
Loaded model: gpt2-medium (355M)
--------------------------------------------------
Total trainable parameters before: 406,286,336
Total trainable parameters after: 0
Total trainable LoRA parameters: 7,898,384
Initial losses
   Training loss: 3.7684114456176756
   Validation loss: 3.7619335651397705
Ep 1 (Step 000000): Train loss 2.509, Val loss 2.519
...
Ep 2 (Step 000230): Train loss 0.308, Val loss 0.652
...
--------------------------------------------------
Generating responses
100% 110/110 [01:52<00:00,  1.03s/it]
Responses saved as instruction-data-with-response-lora.json
Model saved as gpt2-medium355M-sft-lora.pth
```

Karşılaştırma için, özgün 7. bölüm ince ayar kodunu `python exercise_experiments.py --exercise_solution baseline` ile çalıştırabilirsiniz. 

Bir Nvidia L4 GPU'da, LoRA kullanan yukarıdaki kodun 1,30 dakikada çalıştığını unutmayın. Buna karşılık temel çizgi 1,80 dakika sürüyor. Yani LoRA yaklaşık %28 daha hızlı.


Başarımı Ollama Llama 3 yöntemiyle değerlendirebiliriz; kolaylık olsun diye bu da `python exercise_experiments.py` betiğinde uygulanmıştır ve şöyle çalıştırılabilir:

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-lora.json
```

Çıktı:

```
Ollama running: True
Scoring entries: 100%|████████████████████████| 110/110 [01:13<00:00,  1.50it/s]
Number of scores: 110 of 110
Average score: 50.23
```

Puan 50 civarında; bu da özgün modelle aynı aralıkta.